# 语言模型和数据集

In [1]:
import random
import torch
from d2l import torch as d2l
import re

d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')


def read_time_machine():  #@save
    """将时间机器数据集加载到文本行的列表中"""
    with open(d2l.download('time_machine'), 'r') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]


tokens = d2l.tokenize(read_time_machine())
corpus = [token for line in tokens for token in line]
vocab = d2l.Vocab(corpus)
len(vocab),len(corpus)

(4580, 32775)

In [2]:
vocab.token_freqs[:10]

[('the', 2261),
 ('i', 1267),
 ('and', 1245),
 ('of', 1155),
 ('a', 816),
 ('to', 695),
 ('was', 552),
 ('in', 541),
 ('that', 443),
 ('my', 440)]

In [3]:
freqs = [freq for token,freq in vocab.token_freqs]
d2l.plot(freqs, xlabel='token: x', ylabel='frequency: n(x)',
         xscale='log', yscale='log')

<Figure size 350x250 with 1 Axes>

In [4]:
# 将相邻的两个词用空格连接成字符串，作为bigram token
bigram_tokens = [' '.join(pair) for pair in zip(corpus[:-1], corpus[1:])]
bigram_vocab = d2l.Vocab([bigram_tokens])
bigram_vocab.token_freqs[:10]

[('of the', 309),
 ('in the', 169),
 ('i had', 130),
 ('i was', 112),
 ('and the', 109),
 ('the time', 102),
 ('it was', 99),
 ('to the', 85),
 ('as i', 78),
 ('of a', 73)]

In [5]:
# 同样的方法可以构建三元组（trigram）
trigram_tokens = [' '.join([corpus[i], corpus[i+1], corpus[i+2]]) 
                  for i in range(len(corpus)-2)]
trigram_vocab = d2l.Vocab([trigram_tokens])
trigram_vocab.token_freqs[:10]


[('the time traveller', 59),
 ('the time machine', 30),
 ('the medical man', 24),
 ('it seemed to', 16),
 ('it was a', 15),
 ('here and there', 15),
 ('seemed to me', 14),
 ('i did not', 14),
 ('i saw the', 13),
 ('i began to', 13)]

In [6]:
bigram_freqs = [freq for token, freq in bigram_vocab.token_freqs]
trigram_freqs = [freq for token, freq in trigram_vocab.token_freqs]
d2l.plot([freqs, bigram_freqs, trigram_freqs], xlabel='token: x',
         ylabel='frequency: n(x)', xscale='log', yscale='log',
         legend=['unigram', 'bigram', 'trigram'])

<Figure size 350x250 with 1 Axes>

In [7]:
def seq_data_iter_random(corpus, batch_size, num_steps):
    corpus = corpus[random.randint(0, num_steps-1):]
    num_subseqs = (len(corpus) -1) // num_steps
    # initial_indices用于确定每个子序列的起始索引
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    random.shuffle(initial_indices)
    
    def data(pos):
        return corpus[pos:pos+num_steps]
    
    num_batchs = num_subseqs // batch_size
    
    for i in range(0, num_batchs * batch_size, batch_size):
        initial_indices_per_batch = initial_indices[i: i+batch_size]
        X = [data(j) for j in initial_indices_per_batch]
        Y = [data(j+1) for j in initial_indices_per_batch]
        yield torch.tensor(X), torch.tensor(Y)
    

In [8]:
my_seq = list(range(35))
for X,Y in seq_data_iter_random(my_seq, batch_size=2, num_steps=5):
    print('X: ', X, '\nY:', Y)

X:  tensor([[ 9, 10, 11, 12, 13],
        [14, 15, 16, 17, 18]]) 
Y: tensor([[10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])
X:  tensor([[19, 20, 21, 22, 23],
        [24, 25, 26, 27, 28]]) 
Y: tensor([[20, 21, 22, 23, 24],
        [25, 26, 27, 28, 29]])
X:  tensor([[29, 30, 31, 32, 33],
        [ 4,  5,  6,  7,  8]]) 
Y: tensor([[30, 31, 32, 33, 34],
        [ 5,  6,  7,  8,  9]])


In [9]:
def seq_data_iter_sequential(corpus, batch_size, num_steps):
    offset = random.randint(0, num_steps)
    num_tokens = ((len(corpus) - offset - 1) // batch_size) * batch_size
    Xs = torch.tensor(corpus[offset: offset + num_tokens])
    Ys = torch.tensor(corpus[offset+1: offset + num_tokens +1])
    Xs, Ys = Xs.reshape(batch_size, -1), Ys.reshape(batch_size, -1, )
    num_batches = Xs.shape[1] // num_steps
    for i in range(0, num_steps * num_batches, num_steps):
        X = Xs[:, i: i + num_steps]
        Y = Ys[:, i: i + num_steps]
        yield X, Y

In [10]:
for X,Y in seq_data_iter_sequential(my_seq, batch_size=2, num_steps=5):
    print('X: ', X, '\nY:', Y)

X:  tensor([[ 0,  1,  2,  3,  4],
        [17, 18, 19, 20, 21]]) 
Y: tensor([[ 1,  2,  3,  4,  5],
        [18, 19, 20, 21, 22]])
X:  tensor([[ 5,  6,  7,  8,  9],
        [22, 23, 24, 25, 26]]) 
Y: tensor([[ 6,  7,  8,  9, 10],
        [23, 24, 25, 26, 27]])
X:  tensor([[10, 11, 12, 13, 14],
        [27, 28, 29, 30, 31]]) 
Y: tensor([[11, 12, 13, 14, 15],
        [28, 29, 30, 31, 32]])


In [11]:
lines = read_time_machine()
print(len(lines))
print(lines[0])
print(lines[10])

3221
the time machine by h g wells
twinkled and his usually pale face was flushed and animated the


In [12]:
def tokenize(lines, token='word'):
    if token == 'word':
        # split() 默认以任意空白字符（空格、制表符等）分割
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('错误：未知词元类型：' + token)

tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


In [13]:
# 下面是对Vocab类和count_corpus函数的解释性重写，增加中文注释，便于理解其工作原理。

import collections


class Vocab:  #@save
    """
    文本词表类，用于将字符串词元和索引进行双向映射，同时计算和保存词频信息。

    参数:
        tokens: 用于构建词表的词元列表（可以是1D或2D）。
        min_freq: 包含进入词表的最低词频。
        reserved_tokens: 预留的特殊词元，如['<pad>']等。

    成员:
        idx_to_token: 存储每个索引对应的词元（列表）。
        token_to_idx: 存储每个词元对应的索引（字典）。
        _token_freqs: 存储每个词元及其对应词频（降序）。
    """
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:         # 如果没有提供tokens，默认为空列表
            tokens = []
        if reserved_tokens is None: # 如果没有提供预留词元，默认为空列表
            reserved_tokens = []
        # 1. 统计词频，结果是（词元, 词频）元组
        counter = count_corpus(tokens)
        print(counter)
        # 2. 将词频结果按频率降序排序
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)

        # 3. 初始化词元到索引和索引到词元的映射。未知词元'<unk>'的索引为0
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}

        # 4. 添加满足最低词频的高频词元到词表映射中
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break      # 低于最低词频时停止添加
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        """返回词表中词元的数量"""
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        """
        支持用token或token的列表取索引（若为单token返回索引，列表则递归处理）
        未知词元将返回unk索引(通常为0)
        """
        if not isinstance(tokens, (list, tuple)):
            # 返回单个token的索引，若不存在则返回unk
            return self.token_to_idx.get(tokens, self.unk)
        # 对每个token递归取索引
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        """
        支持用索引或索引列表取词元
        """
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):
        """返回未知词元的索引，通常为0"""
        return 0

    @property
    def token_freqs(self):
        """返回(词元, 词频)的有序列表"""
        return self._token_freqs

def count_corpus(tokens):  #@save
    """
    统计语料中各词元的出现频率
    参数:
        tokens: 1D(词元列表)或2D(每行为词元列表的列表)
    返回:
        collections.Counter对象（词元到计数的映射）
    """
    # 若为2D列表则展开为1D列表
    if len(tokens) == 0 or isinstance(tokens[0], list):
        # 将每一行的词元合并到一个大列表中
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)  # 统计每个词元的出现次数

In [14]:
def load_corpus_time_machine(max_tokens=-1):
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    corpus = [vocab[token] for line in tokens for token in line]
    print(corpus)
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

corpus, vocab = load_corpus_time_machine()
len(corpus), len(vocab)

Counter({' ': 29927, 'e': 17838, 't': 13515, 'a': 11704, 'i': 10138, 'n': 9917, 'o': 9758, 's': 8486, 'h': 8257, 'r': 7674, 'd': 6337, 'l': 6146, 'm': 4043, 'u': 3805, 'c': 3424, 'f': 3354, 'w': 3225, 'g': 3075, 'y': 2679, 'p': 2427, 'b': 1897, 'v': 1295, 'k': 1087, 'x': 236, 'z': 144, 'j': 97, 'q': 95})
[3, 9, 2, 1, 3, 5, 13, 2, 1, 13, 4, 15, 9, 5, 6, 2, 1, 21, 19, 1, 9, 1, 18, 1, 17, 2, 12, 12, 8, 5, 3, 9, 2, 1, 3, 5, 13, 2, 1, 3, 10, 4, 22, 2, 12, 12, 2, 10, 1, 16, 7, 10, 1, 8, 7, 1, 5, 3, 1, 17, 5, 12, 12, 1, 21, 2, 1, 15, 7, 6, 22, 2, 6, 5, 2, 6, 3, 1, 3, 7, 1, 8, 20, 2, 4, 23, 1, 7, 16, 
……（冗长输出已省略）


(170580, 28)

In [15]:
class SeqDataLoader:
    def __init__(self, batch_size, num_steps, use_random_iter, max_tokens) -> None:
        if use_random_iter:
            self.data_iter_fn = seq_data_iter_random
        else:
            self.data_iter_fn = seq_data_iter_sequential
        
        self.corpus, self.vocab = load_corpus_time_machine(max_tokens)
        self.batch_size, self.num_steps = batch_size, num_steps
    
    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)

In [16]:
def load_data_time_machine(batch_size, num_steps, use_random_iter=False, max_tokens=10000):
    data_iter = SeqDataLoader(batch_size, num_steps, use_random_iter, max_tokens)
    yield data_iter, data_iter.vocab